# v5固定+0.10等beta双臂来源支持

只读逐层点时父本及Qlib分钟来源；两臂必须同一完成bar可在t+2整股扣Tiered成交。零未来收益、零拟合、零候选账户。

In [1]:
import os
os.environ["MLFLOW_ALLOW_FILE_STORE"]="true"
os.environ["MLFLOW_DISABLE_AGENT_HINT"]="1"
import hashlib,json,warnings
from copy import deepcopy
from pathlib import Path
import exchange_calendars as xc
import numpy as np
import pandas as pd
import qlib
from qlib.data import D
from src.workflow.minute_action_value import _source
from src.backtest.event_guard_clock import guard_slot
from src.backtest.ibkr_tiered import TieredFees,settle_tiered_event
warnings.filterwarnings("ignore",message="The 'generic' unit for NumPy timedelta")
root=Path.cwd();card=root/"research/stage1_v5_fixed_equal_beta_support_20261030"
raw_plan=(card/"plan.json").read_bytes();plan=json.loads(raw_plan)
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert sha(root/plan["provider_uri"]/"qlib_store_manifest.json")==plan["provider_manifest_sha256"]
qlib.init(provider_uri=str(root/plan["provider_uri"]),region="us",kernels=1)
source,source_receipt=_source(root,plan["training_source"],["analysis/native/dated_f696_0"])
frame_source,frame_receipt=_source(root,plan["frame_source"],["inputs/covered_frame"])
book=source["analysis/native/dated_f696_0"];frame=frame_source["inputs/covered_frame"].reset_index(drop=True)
assert len(frame)==plan["expected_parent_sessions"]==len(book["authority"])
assert frame.trade_date.equals(book["authority"].trade_date)
assert book["audit"]["status"]=="passed" and book["audit"]["slippage_bps"]==0
bars=D.features(["SOXL","SOXX"],["$open","$close","$raw_open","$price_valid"],start_time=plan["training_start"]+" 09:31:00",end_time=plan["training_end"]+" 16:00:00",freq="1min")
parts=[]
for symbol in ("SOXL","SOXX"):
    part=bars.xs(symbol,level="instrument").copy()
    part.columns=[symbol+"_"+field.removeprefix("$") for field in part]
    parts.append(part)
quotes=pd.concat(parts,axis=1).sort_index()
schedule=xc.get_calendar("XNYS",start=frame.trade_date.min(),end=frame.trade_date.max()).schedule
closes={pd.Timestamp(day).tz_localize(None):pd.Timestamp(row.close).tz_convert("America/New_York").tz_localize(None) for day,row in schedule.iterrows()}
orders=book["orders"].sort_values("stamp")
by_day={pd.Timestamp(day):part for day,part in orders.groupby("trade_date",sort=True)}
fees=TieredFees(plan["tariff"],slippage_bps=0)
records=[];scanned=0;source_slots=0
for index,row in frame.iterrows():
    day=pd.Timestamp(row.trade_date)
    current=by_day.get(day,pd.DataFrame())
    asset=quotes.loc[quotes.index.normalize()==day]
    status=("terminal" if index+1>=len(frame) else
            "original_recovery" if bool(book["authority"].iloc[index].recovery_action) else
            "source_invalid" if asset.empty or not asset[["SOXL_price_valid","SOXX_price_valid"]].eq(1).all().all()
            else "eligible")
    record={"trade_date":str(day.date()),"year":int(day.year),"status":status,"source_slots":0,
            "tested_slots":0,"both_funded":False,"first_funded_bar_end_at":None}
    if status=="eligible":
        opened=day+pd.Timedelta(hours=9,minutes=47)
        position=book["native_positions"].get(opened)
        assert position is not None
        q=np.array([position.get_stock_amount(symbol) if position.check_stock(symbol) else 0. for symbol in ("SOXL","SOXX")],float)
        cash=float(position.get_cash())
        close=closes[day]
        eligible=asset.loc[(asset.index>=opened+pd.Timedelta(minutes=1)) & (asset.index+pd.Timedelta(minutes=2)<close)]
        record["source_slots"]=len(eligible);source_slots+=len(eligible)
        branch=deepcopy(fees)
        for order in current.itertuples(index=False):
            if order.slot=="open":branch.commit(int(order.raw_shares),stamp=order.stamp,cap_applied=bool(order.cap_applied))
        for minute in eligible.itertuples():
            scanned+=1;record["tested_slots"]+=1
            slot=guard_slot(minute.Index,session_open=day+pd.Timedelta(hours=9,minutes=30),session_close=close,original_open_fill=opened)
            assert slot is not None
            mark=np.array([minute.SOXL_close,minute.SOXX_close],float)
            nav=float(q@mark+cash)
            if not np.isfinite(nav) or nav<=0:continue
            weights=np.r_[q*mark,cash]/nav
            if weights[2]<.10-1e-10:continue
            targets={"SOXX":weights+np.array([0.,.10,-.10]),"SOXL":weights+np.array([.10/3,0.,-.10/3])}
            if any(target.min() < -1e-10 or abs(target.sum()-1)>1e-8 for target in targets.values()):continue
            fillbar=asset.loc[slot.fill_at+pd.Timedelta(minutes=1)]
            fill=np.array([fillbar.SOXL_open,fillbar.SOXX_open],float)
            raw=np.array([fillbar.SOXL_raw_open,fillbar.SOXX_raw_open],float)
            funded={}
            for symbol,target in targets.items():
                trade=settle_tiered_event(q,cash,desired=target,adjusted_prices=fill,raw_prices=raw,stamp=slot.fill_at,fees=deepcopy(branch))
                after_nav=float(trade["quantities"]@fill+trade["cash"])
                order_now=trade["orders"]
                funded[symbol]=bool(after_nav>0 and trade["cash"]>=-1e-8 and
                    len(order_now)==1 and order_now[0]["symbol"]==symbol and not order_now[0]["sell"] and
                    (trade["quantities"]>=0).all())
            if funded["SOXL"] and funded["SOXX"]:
                record["both_funded"]=True;record["first_funded_bar_end_at"]=str(slot.bar_end_at)
                break
    records.append(record)
    for order in current.itertuples(index=False):
        fees.commit(int(order.raw_shares),stamp=order.stamp,cap_applied=bool(order.cap_applied))
assert scanned<=source_slots<=plan["budget"]["maximum_event_slots"]
perday=pd.DataFrame(records)
annual=perday.groupby("year",sort=True).agg(parent_days=("trade_date","size"),source_eligible_days=("status",lambda s:int(s.eq("eligible").sum())),source_slots=("source_slots","sum"),both_funded_days=("both_funded","sum")).reset_index()
gate=plan["support_gate"]
counts={int(r.year):int(r.both_funded_days) for r in annual.itertuples(index=False)}
passed=bool(sum(counts.get(y,0) for y in (2018,2019,2020))>=gate["minimum_discovery_days_2018_2020"] and
            counts.get(2019,0)>=gate["minimum_2019_days"] and counts.get(2020,0)>=gate["minimum_2020_days"] and
            counts.get(2021,0)>=gate["minimum_2021_days"] and counts.get(2022,0)>=gate["minimum_2022_days"])
(card/"annual_support.csv").write_text(annual.to_csv(index=False))
(card/"perday_support.csv").write_text(perday.to_csv(index=False))
audit={"experiment":plan["experiment"],"plan_sha256":hashlib.sha256(raw_plan).hexdigest(),"decision":"fixed_010_source_support_pass_needs_value_labels" if passed else "not_admitted_fixed_010_source_support_failed",
       "source_recorder_ids":{"parent":plan["training_source"]["recorder_id"],"frame":plan["frame_source"]["recorder_id"]},"source_readback_entries":{"parent":len(source_receipt),"frame":len(frame_receipt)},
       "parent_sessions":len(frame),"source_slots":source_slots,"scanned_until_first_funded":scanned,"annual_funded_days":counts,"support_gate_pass":passed,
       "new_model_fits":0,"new_native_accounts":0,"future_returns_read":0,"historical_research_passed":False,
       "limits":["One event per day first funded slot is for support counting only, not a trading trigger.","Historical minute first receipt and actual fill/quote response remain unqualified.","No value labels or SOXX account result."]}
(card/"audit.json").write_text(json.dumps(audit,ensure_ascii=False,indent=2)+"\n")
print(json.dumps({"decision":audit["decision"],"annual_funded_days":counts,"source_slots":source_slots,"scanned":scanned},ensure_ascii=False))

## 中文结论

1056父本日/384660来源分钟槽上固定+0.10名义beta两买入臂，各自整股含Tiered且同一分钟可行动，2018–2022分别53/252/242/252/245日；发现期547日、2021–2022检验期497日，来源支持门通过。零未来收益标签、拟合、候选账户与SOXX超额。首个可行动分钟只用于计数；原恢复日10个被事后屏蔽，不能用于实盘提前筛选。源无效1日、终端1日未知。收入和状态样本支持尚未检验，下一步三臂费后数值标签及2018–2020控制残差发现。NAS父本和分钟来源已限定读回；本卡无新Recorder。